# Notebook 01 - Neural Networks

Goes with **Lesson 01**. Run each cell with **Shift + Enter**, from top to bottom.

## 1. One neuron by hand: "Should I go out to play?"

A neuron = multiply inputs by weights, add them up with a bias, then apply an activation.

In [ ]:
import numpy as np

inputs = np.array([1, 0, 1])      # sunny = yes, homework finished = no, very hot = yes
weights = np.array([3, 4, -2])    # how important each input is
bias = -1

total = np.dot(inputs, weights) + bias
print("Weighted sum + bias =", total)

def step(x):
    return 1 if x > 0 else 0

print("Decision:", "go and play" if step(total) == 1 else "stay home")

**Try it:** change homework finished to `1` and run again.

## 2. Activation functions

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-x))
relu = np.maximum(0, x)
tanh = np.tanh(x)

plt.figure(figsize=(9, 4))
plt.plot(x, sigmoid, label="sigmoid (0 to 1)")
plt.plot(x, relu, label="ReLU (negative -> 0)")
plt.plot(x, tanh, label="tanh (-1 to 1)")
plt.ylim(-1.5, 3)
plt.axhline(0, color="gray", linewidth=0.5)
plt.title("Activation functions")
plt.legend()
plt.show()

scores = np.array([2.0, 1.0, 0.1])
softmax = np.exp(scores) / np.exp(scores).sum()
print("Softmax turns scores", scores, "into probabilities", softmax.round(3), "which add up to", softmax.sum())

## 3. Train one neuron with gradient descent

We teach one neuron the **AND** rule: output 1 only when **both** inputs are 1.
The neuron starts with random weights and improves them over many **epochs**.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

def train_one_neuron(X, y, epochs=3000, learning_rate=0.5, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.normal(size=X.shape[1])
    b = 0.0
    losses = []
    for epoch in range(epochs):
        p = sigmoid_fn(X @ w + b)                                     # forward pass
        loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))   # cross-entropy loss
        losses.append(loss)
        error = p - y                                                 # how wrong
        w -= learning_rate * (X.T @ error) / len(y)                   # update weights
        b -= learning_rate * error.mean()                             # update bias
    return w, b, losses

w, b, losses = train_one_neuron(X, y_and)
print("Learned weights:", w.round(2), " bias:", round(b, 2))
print("Predictions for AND:", (sigmoid_fn(X @ w + b) > 0.5).astype(int), " correct answers:", y_and)

plt.plot(losses)
plt.title("Loss going down while one neuron learns AND")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 4. One neuron is not enough: the XOR problem

**XOR** = output 1 when the inputs are **different**. You cannot separate XOR with one straight line, so one neuron fails.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
w, b, _ = train_one_neuron(X, y_xor)
pred = (sigmoid_fn(X @ w + b) > 0.5).astype(int)
print("One neuron predicts:", pred, " correct answers:", y_xor)
print("Accuracy:", (pred == y_xor).mean())

## 5. Add a hidden layer and it works

A tiny network: 2 inputs -> **4 hidden neurons** -> 1 output. We write the forward pass and **backpropagation** ourselves.

In [ ]:
def train_small_network(X, y, hidden=4, epochs=5000, learning_rate=0.5, seed=1):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=(2, hidden)); b1 = np.zeros(hidden)
    W2 = rng.normal(size=(hidden, 1)); b2 = np.zeros(1)
    y = y.reshape(-1, 1)
    losses = []
    for epoch in range(epochs):
        # forward pass
        h = np.tanh(X @ W1 + b1)              # hidden layer
        p = sigmoid_fn(h @ W2 + b2)           # output layer
        losses.append(-np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9)))
        # backpropagation: send the error backwards, layer by layer
        d_out = (p - y) / len(y)
        d_W2 = h.T @ d_out;  d_b2 = d_out.sum(axis=0)
        d_h = (d_out @ W2.T) * (1 - h ** 2)
        d_W1 = X.T @ d_h;    d_b1 = d_h.sum(axis=0)
        # update every weight a little (gradient descent)
        W1 -= learning_rate * d_W1; b1 -= learning_rate * d_b1
        W2 -= learning_rate * d_W2; b2 -= learning_rate * d_b2
    predict = lambda data: (sigmoid_fn(np.tanh(data @ W1 + b1) @ W2 + b2) > 0.5).astype(int).ravel()
    return predict, losses

predict, losses = train_small_network(X, y_xor, hidden=4)
print("Network with a hidden layer predicts:", predict(X), " correct answers:", y_xor)

plt.plot(losses)
plt.title("XOR: loss with one hidden layer")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

The hidden layer lets the network bend its decision boundary. This is why **deep** networks (many layers) can learn very complex patterns.

## 6. A real neural network: reading handwritten digits

scikit-learn includes 1,797 tiny images (8 x 8 pixels) of handwritten digits 0 to 9. No download needed.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("Images:", digits.images.shape, " Labels:", digits.target.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

print("One image is just 64 numbers (0 = white, 16 = black):")
print(digits.images[0].astype(int))

We train an **MLP** (Multi-Layer Perceptron = a basic neural network):

- input layer: 64 pixels,
- one hidden layer with 64 neurons (ReLU),
- output layer: 10 neurons (softmax, one per digit),
- `batch_size=32`: update the weights after every 32 images,
- `max_iter=300`: up to 300 epochs (it stops earlier when the loss stops improving).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_digits = digits.data / 16.0          # scale pixels to 0..1
X_train, X_test, y_train, y_test = train_test_split(X_digits, digits.target, test_size=0.2, random_state=42)

network = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", batch_size=32,
                        max_iter=300, random_state=42)
network.fit(X_train, y_train)

print(f"Test accuracy: {network.score(X_test, y_test):.3f}")
print("Epochs trained:", network.n_iter_)

plt.plot(network.loss_curve_)
plt.title("Training loss per epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
predictions = network.predict(X_test[:10])

fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for ax, pixels, pred, real in zip(axes, X_test[:10], predictions, y_test[:10]):
    ax.imshow(pixels.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"pred {pred}", color="green" if pred == real else "red")
    ax.axis("off")
plt.show()

Green = correct, red = wrong.

**Try it:** change `hidden_layer_sizes=(64,)` to `(16,)` and to `(128, 64)` (two hidden layers).

Go back to **Lesson 01** and do the **Check** section.